In [ ]:
# everything connected to directorys is my own preference and can be altered 
# "xxx" is a placeholder for the name of the array detection prgramm run in the first place

# CRISPRidentify

**I used it in a terminal:**  
python CRISPRidentify.py --file /home/path/to/input_dir/arrays/**xxx**_R1_und_R2.fasta --result_folder /home/path/to/input_dir/CRISPRidentify/**xxx**/ --min_len_rep 23 --max_len_rep 50 --min_repeats 3 --min_len_spacer 23 --max_len_spacer 50

## Cleanup

In [ ]:
cd /home/path/to/input_dir/CRISPRidentify/xxx/
find . -type f ! -name "Complete_summary.csv" -delete
cd /home/path/to/working_dir/

## Filtering

In [ ]:
awk -F',' '
    $13=="Bona-Fide" || $13=="Alternative" || $13=="Possible" {
        if (match($1, /Group-([0-9]+)/, arr)) {
            print "Group\\|" arr[1] "(\\||$)"
        }
    }
' /home/path/to/input_dir/CRISPRidentify/xxx/Complete_summary.csv > /home/path/to/input_dir/Identified_Groups.txt

**I used it in a in terminal:**  
seqkit grep -w 0 -r -f /home/path/to/input_dir/Identified_Groups.txt /home/path/to/input_dir/repeats_und_spacer/**xxx**_R1_und_R2.fasta > /home/path/to/input_dir/CRISPRidentify/**xxx**/CRISPR_identified_sequences.fasta

<br>
<br>

# CRISPRDetect

**I used it in a terminal:**  
perl CRISPRDetect.pl -f /home/path/to/input_dir/arrays/**xxx**_R1_und_R2.fasta -o /home/path/to/input_dir/CRISPRDetect/**xxx**/CRISPRDetect -check_direction 0 -array_quality_score_cutoff 3 -T 5 -minimum_repeat_length 23 -minimum_no_of_repeats 3

## Filtering

In [ ]:
CRISPR_OUTPUT="/home/path/to/input_dir/CRISPRDetect/xxx/CRISPRDetect"
grep -oP 'Group\|\d+' "$CRISPR_OUTPUT" | sed 's/[[:space:]]*$//; s/|$//; s/|/\\|/g; s/.*/&(\\||$)/' > /home/path/to/input_dir/Detected_Groups.txt

**I used it in a in terminal:**   
seqkit grep -w 0 -r -f /home/path/to/input_dir/Detected_Groups.txt /home/path/to/input_dir/repeats_und_spacer/**xxx**_R1_und_R2.fasta > /home/path/to/input_dir/CRISPRDetect/**xxx**/CRISPR_detected_sequences.fasta

<br>
<br>

# Preparation for BLAST and CRISPRclassify

In [ ]:
awk '/^>DR/ {print; getline; print}
' /home/path/to/input_dir/repeats_und_spacer/xxx_R1_und_R2.fasta > /home/path/to/output_dir/CRISPRclassify/xxx/repeats_R1_und_R2.fasta
awk '/^>DR/ {print; getline; print}
' /home/path/to/input_dir/repeats_und_spacer/xxx_R1_und_R2.fasta > /home/path/to/output_dir/BLAST/xxx/repeats_R1_und_R2.fasta
awk '/^>SP/ {print; getline; print}
' /home/path/to/input_dir/repeats_und_spacer/xxx_R1_und_R2.fasta > /home/path/to/output_dir/BLAST/xxx/spacer_R1_und_R2.fasta
awk 'NR%2==0 {print $1}
' /home/path/to/input_dir/CRISPRclassify/xxx/repeats_R1_und_R2.fasta > /home/path/to/output_dir/CRISPRclassify/xxx/repeats_xxx.txt

<br>
<br>

# BLAST
## Preperation Part 2

In [ ]:
cd /home/path/to/input_dir/BLAST/xxx

In [ ]:
mkdir temp_spacer temp_repeats temp_db
awk '
/^>/ {
    split($0,a,"|")
    group=a[3]
    file="temp_spacer/group|"group".fasta"
    close(file)
}
{ print >> file }
' spacer_R1_und_R2.fasta

awk '
/^>/ {
    split($0,a,"|")
    group=a[3]
    file="temp_repeats/group|"group".fasta"
    close(file)
}
{ print >> file }
' repeats_R1_und_R2.fasta

## DB creation and BLAST run

In [ ]:
rm -f spacer_repeat_match_same_Group.tab
rm -f temp_db/makeblastdb.log 

time for f in temp_spacer/group\|*.fasta
do
    group=$(basename "$f" .fasta | cut -d"|" -f2)

    if [ -f "temp_repeats/group|${group}.fasta" ]; then
        
        makeblastdb -in "temp_repeats/group|${group}.fasta" \
                    -dbtype nucl \
                    -out "temp_db/db|${group}" \
                    >> temp_db/makeblastdb.log 

        blastn \
        -db "temp_db/db|${group}" \
        -query "$f" \
        -task blastn-short \
        -outfmt "6 qseqid sseqid pident length qlen slen evalue bitscore" \
        -evalue 1000000 \
        -word_size 4 \
        -dust no \
        -soft_masking false \
        >> spacer_repeat_match_same_Group.tab
    fi
done

## Apply Filter

In [ ]:
awk '{ 
    pident=$3; len=$4; qlen=$5; slen=$6;
    # Check: Identität >= 80% UND Länge des Alignments mindestens 80 % des spacers (kein 4 basen alignment bei sequenzen die 20 basen haben)
    if (pident >= 80 && (len >= (qlen * 0.80) || len >= (slen * 0.80))) {
        print $1
    }
}' spacer_repeat_match_same_Group.tab > blacklisted_spacers.txt
awk '{
        if (match($1, /Group\|([0-9]+)/, arr)) {
            print "Group\\|" arr[1] "(\\||$)"
        }
    }
' blacklisted_spacers.txt | sort -u > /home/path/to/output_dir/Blasted_Groups.txt

**I used it in a in terminal:**   
conda run -n seqkit_env seqkit grep -w 0 -v -r -f /home/path/to/input_dir/Blasted_Groups.txt /home/path/to/input_dir/repeats_und_spacer/**xxx**_R1_und_R2.fasta > /home/path/to/output_dir/BLAST/**xxx**/CRISPR_Blast_filtered_sequences.fasta

## Cleanup

In [ ]:
cd /home/path/to/input_dir/BLAST/xxx
rm -rf temp_spacer temp_repeats temp_db
cd /home/path/to/working_dir

<br>
<br>

# CRISPRclassify
## Preparation Part 2

In [ ]:
awk '/^>DR/ {print; getline; print}
' /home/path/to/input_dir/BLAST/xxx/CRISPR_Blast_filtered_sequences.fasta > /home/path/to/output_dir/CRISPRclassify/xxx_blasted/repeats_R1_und_R2.fasta

awk 'NR%2==0 {print $1}
' /home/path/to/input_dir/CRISPRclassify/xxx_blasted/repeats_R1_und_R2.fasta > /home/path/to/output_dir/CRISPRclassify/xxx_blasted/repeats_xxx_blasted.txt

**Save following Code as CRISPRclassify.R:**

In [ ]:
start_time <- Sys.time()
cat("start\n")

system(sprintf("taskset -cp 0-5 %d > /dev/null 2>&1", Sys.getpid()))
# ---------------- Get Arguments ----------------

args <- commandArgs(trailingOnly = TRUE)

getArg <- function(flag) {
  idx <- which(args == flag)
  if(length(idx) == 0) stop(paste("Missing argument:", flag))
  args[idx + 1]
}
getArgOptional <- function(flag, default = NA) {
  idx <- which(args == flag)
  if(length(idx) == 0) return(default)
  args[idx + 1]
}

input_file <- getArg("-input")
max_repeats <- as.numeric(getArgOptional("-max_repeats", Inf))


#------------ CRISPRclassify Run ------------ 

suppressPackageStartupMessages(library(CRISPRclassify))
invisible(capture.output(
  CRISPRclassify::classifyRepeats(input_file),
  type = "output"
))

cat("classified\n")


#------------ Sort ------------ 

Vorlage <- read.table(input_file, 
                      stringsAsFactors=FALSE) 
colnames(Vorlage) <- "Repeat" 

classified <- read.table(paste0(input_file, ".crclass"), 
                         header=TRUE, 
                         sep=',') 

classify_sorted <- classified[match(Vorlage$Repeat, classified$Repeat), ] 

rownames(classify_sorted) <- NULL 

  
  fasta_file <- file.path(dirname(input_file), "repeats_R1_und_R2.fasta")
  
  fasta_lines <- readLines(fasta_file)
  headers <- fasta_lines[grepl("^>", fasta_lines)]
  
  groups <- sub("^>DR\\|Group\\|([0-9]+)\\|.*", "\\1", headers)
  
  classify_sorted <- cbind(Group = groups, classify_sorted)
  

base <- basename(input_file)
prefix <- sub("^repeats_(.*)\\.txt$", "\\1", base)

out_sorted <- file.path(dirname(input_file),
                        paste0(prefix, "_classified_repeats.csv"))
write.csv(classify_sorted, 
          out_sorted, 
          row.names = FALSE) 
cat("sorted\n")


#------------ End ------------ 

end_time <- Sys.time()
cat("done\n")
cat("Time:", round(difftime(end_time, start_time, units = "secs"), 2), "Seconds\n")

## Run CRISPRclassify with Rscript

In [ ]:
Classify() { Rscript /home/path/to/CRISPRclassify.R "$@"; }

In [ ]:
Classify -input /home/path/to/input_dir/CRISPRclassify/xxx/repeats_xxx.txt # -max_repeats

In [ ]:
Classify -input /home/path/to/input_dir/CRISPRclassify/xxx_blasted/repeats_xxx_blasted.txt # -max_repeats

## Filter CRISPRclassify

### By Edit Distance
#### Save following Code as Filter_Dist.R:

In [ ]:
# ---------------- Get Arguments ----------------

args <- commandArgs(trailingOnly = TRUE)

getArg <- function(flag) {
  idx <- which(args == flag)
  if(length(idx) == 0) stop(paste("Missing argument:", flag))
  args[idx + 1]
}
getArgOptional <- function(flag, default = NA) {
  idx <- which(args == flag)
  if(length(idx) == 0) return(default)
  args[idx + 1]
}
input_file <- getArg("-input")
dist_cutoff <- as.numeric(getArg("-dist"))
prob_cutoff <- as.numeric(getArg("-prob"))
max_repeats <- as.numeric(getArgOptional("-max_repeats", Inf))


#------------ Sort ------------

Vorlage <- read.table(input_file, 
                      stringsAsFactors=FALSE) 
colnames(Vorlage) <- "Repeat" 

classified <- read.table(paste0(input_file, ".crclass"), 
                         header=TRUE, 
                         sep=',') 

classify_sorted <- classified[match(Vorlage$Repeat, classified$Repeat), ] 

rownames(classify_sorted) <- NULL 


fasta_file <- file.path(dirname(input_file), "repeats_R1_und_R2.fasta")

fasta_lines <- readLines(fasta_file)
headers <- fasta_lines[grepl("^>", fasta_lines)]

groups <- sub("^>DR\\|Group\\|([0-9]+)\\|.*", "\\1", headers)

classify_sorted <- cbind(Group = groups, classify_sorted)


#------------ Filter ------------ 

dist_values <- classify_sorted[,10]
prob_values <- classify_sorted[,4]

Dist_Greater_X <- classify_sorted[
  (!is.na(dist_values) & dist_values < dist_cutoff) |
    (is.na(dist_values) & prob_values > prob_cutoff),
]

Dist_Greater_X <- Dist_Greater_X[order(Dist_Greater_X[,10]), ]

Dist_Greater_X <- Dist_Greater_X[seq_len(min(nrow(Dist_Greater_X), max_repeats)), ]

out_filtered <- file.path(dirname(input_file),
                          paste0("Dist_", dist_cutoff, "_filtered_and_classified_repeats.csv"))
write.csv(Dist_Greater_X, 
          out_filtered, 
          row.names = FALSE)

#### Run Filter_Dist with Rscript

In [ ]:
Filter_Dist() { Rscript /home/path/to/Filter_Dist.R $@; }

In [ ]:
Filter_Dist -input /home/path/to/input_dir/CRISPRclassify/xxx/repeats_xxx.txt -dist 9 -prob 0.975 # -max_repeats

In [ ]:
Filter_Dist -input /home/path/to/input_dir/CRISPRclassify/xxx_blasted/repeats_xxx_blasted.txt -dist 9 -prob 0.975 # -max_repeats

### By Probability
#### Save following Code as Filter_Prob.R:

In [ ]:
# ---------------- Get Arguments ----------------

args <- commandArgs(trailingOnly = TRUE)

getArg <- function(flag) {
  idx <- which(args == flag)
  if(length(idx) == 0) stop(paste("Missing argument:", flag))
  args[idx + 1]
}
getArgOptional <- function(flag, default = NA) {
  idx <- which(args == flag)
  if(length(idx) == 0) return(default)
  args[idx + 1]
}
input_file <- getArg("-input")
prob_cutoff <- as.numeric(getArg("-prob"))
max_repeats <- as.numeric(getArgOptional("-max_repeats", Inf))


#------------ Sort ------------

Vorlage <- read.table(input_file, 
                      stringsAsFactors=FALSE) 
colnames(Vorlage) <- "Repeat" 

classified <- read.table(paste0(input_file, ".crclass"), 
                         header=TRUE, 
                         sep=',') 

classify_sorted <- classified[match(Vorlage$Repeat, classified$Repeat), ] 

rownames(classify_sorted) <- NULL 


fasta_file <- file.path(dirname(input_file), "repeats_R1_und_R2.fasta")

fasta_lines <- readLines(fasta_file)
headers <- fasta_lines[grepl("^>", fasta_lines)]

groups <- sub("^>DR\\|Group\\|([0-9]+)\\|.*", "\\1", headers)

classify_sorted <- cbind(Group = groups, classify_sorted)


#------------ Filter ------------ 

Prob_Greater_X <- classify_sorted[classify_sorted[,4] > prob_cutoff, ] 

Prob_Greater_X <- Prob_Greater_X[order(-Prob_Greater_X[,4]), ]

Prob_Greater_X <- Prob_Greater_X[seq_len(min(nrow(Prob_Greater_X), max_repeats)), ]

prob_str <- sprintf("%.3f", prob_cutoff)

out_filtered <- file.path(dirname(input_file),
                          paste0("Prob_", prob_str, "_filtered_and_classified_repeats.csv"))

write.csv(Prob_Greater_X, 
          out_filtered, 
          row.names = FALSE)

#### Run Filter_Prob wit Rscript

In [ ]:
Filter_Dist() { Rscript /home/path/to/Filter_Prob.R $@; }

In [ ]:
Filter_Prob -input /home/path/to/input_dir/CRISPRclassify/xxx/repeats_xxx.txt -prob 0.975 # -max_repeats

In [ ]:
Filter_Prob -input /home/path/to/input_dir/CRISPRclassify/xxx_blasted/repeats_xxx_blasted.txt -prob 0.975 # -max_repeats